# Delta Variants: Cosine, Eder, and the MFW Question

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* Comparing Delta measures and MFW bands; bootstrap consensus trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09d_delta_variants.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Notebook 09b implemented Burrows's Delta and then, in an exercise prompt,
mentioned that Cosine Delta usually does better — without ever computing it.
This notebook settles that empirically.

Evert et al. (2017), "Understanding and explaining Delta", found two things the
course should not just assert:

1. **Cosine Delta** is the most robust variant, and the reason is vector
   normalisation rather than the distance metric as such.
2. Around **3,000 most-frequent words** is a robust setting across languages —
   far more than the 100–150 most tutorials use.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["tokens"] = works["tokens"] = works["text"].map(tokenise)
works["N"] = works["tokens"].map(len)
works = works[works["N"] >= 5000].reset_index(drop=True)

counts = works["author"].value_counts()
multi = counts[counts >= 2].index.tolist()
works = works[works["author"].isin(multi)].reset_index(drop=True)
print(f"{len(works)} works by {works['author'].nunique()} authors with 2+ works each")
print(works["author"].value_counts().to_string())

## One feature matrix, three distances

All three variants start from the same z-scored most-frequent-word rates. They
differ only in how distance is measured in that space.

- **Burrows's Delta** — mean absolute difference (Manhattan).
- **Eder's Delta** — Manhattan, but down-weighting lower-ranked words.
- **Cosine Delta** — the angle between vectors, which normalises away magnitude.

In [ ]:
corpus_counts = Counter()
tokenised = []
for t in works["tokens"]:
    tokenised.append(t)
    corpus_counts.update(t)

def feature_matrix(mfw):
    feats = [w for w, _ in corpus_counts.most_common(mfw)]
    rates = np.array([[toks.count(w) / len(toks) for w in feats]
                      for toks in tokenised])
    mu, sd = rates.mean(axis=0), rates.std(axis=0)
    sd[sd == 0] = 1.0
    return (rates - mu) / sd

def burrows(Z):
    return np.abs(Z[:, None, :] - Z[None, :, :]).mean(axis=2)

def eder(Z):
    n = Z.shape[1]
    w = (n - np.arange(n)) / n          # rank-decreasing weights
    return (np.abs(Z[:, None, :] - Z[None, :, :]) * w).sum(axis=2) / w.sum()

def cosine_delta(Z):
    norm = Z / np.linalg.norm(Z, axis=1, keepdims=True)
    return 1 - norm @ norm.T

print("Three distance functions defined over the same z-scored MFW space.")

## Which variant attributes best, and at which MFW setting?

The test is leave-one-out: hide a work, find its nearest neighbour, and ask
whether that neighbour is by the same author. This is the standard way the
stylometry literature compares Delta variants, and it is what produces
Evert's 3,000-word finding.

In [ ]:
authors = works["author"].to_numpy()

def loo_accuracy(D):
    np.fill_diagonal(D, np.inf)
    nearest = D.argmin(axis=1)
    return float((authors[nearest] == authors).mean())

bands = [100, 300, 600, 1000, 2000, 3000, 5000]
results = []
for mfw in bands:
    Z = feature_matrix(mfw)
    results.append({"MFW": mfw,
                    "Burrows": loo_accuracy(burrows(Z)),
                    "Eder": loo_accuracy(eder(Z)),
                    "Cosine": loo_accuracy(cosine_delta(Z))})
res = pd.DataFrame(results)
print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
plt.figure(figsize=(7, 4))
for col, style in [("Burrows", "-o"), ("Eder", "-s"), ("Cosine", "-^")]:
    plt.plot(res["MFW"], res[col], style, label=col)
plt.xscale("log")
plt.xlabel("most frequent words used")
plt.ylabel("leave-one-out attribution accuracy")
plt.title("Delta variants across MFW bands")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()

best = res.set_index("MFW")[["Burrows", "Eder", "Cosine"]].stack().idxmax()
print(f"Best here: {best[1]} Delta at {best[0]} MFW")

## Bootstrap consensus trees

A single dendrogram at a single MFW setting is fragile — change the setting and
the tree rearranges. Eder's answer, and the reason `stylo` ships it, is to build
trees across many MFW bands and keep only the groupings that survive most of
them.

In [ ]:
pair_support = np.zeros((len(works), len(works)))
n_runs = 0
for mfw in [100, 200, 400, 800, 1600, 3000]:
    Z = feature_matrix(mfw)
    D = cosine_delta(Z)
    np.fill_diagonal(D, 0)
    Zl = linkage(squareform(D, checks=False), method="average")
    labels = fcluster(Zl, t=works["author"].nunique(), criterion="maxclust")
    pair_support += (labels[:, None] == labels[None, :])
    n_runs += 1
consensus = pair_support / n_runs

Zc = linkage(squareform(1 - consensus, checks=False), method="average")
plt.figure(figsize=(9, 5))
dendrogram(Zc, labels=[f"{a[:14]}" for a in works["author"]],
           leaf_font_size=7, orientation="right")
plt.title(f"Bootstrap consensus over {n_runs} MFW bands (Cosine Delta)")
plt.xlabel("1 - proportion of runs clustered together")
plt.tight_layout()

### 🔧 Your turn 1

Add 10000 to the `bands` list and re-run. Does accuracy keep climbing, and what does that say about Evert's 3,000-word advice?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Accuracy usually plateaus and can dip: beyond a few thousand words you are
adding items that appear a handful of times in a handful of works, and their
z-scores are dominated by noise. Evert's ~3,000 is not a magic number, it is the
point where the curve has flattened — the useful reading is "use far more than
150, and check where your own curve stops rising", which is a claim you can only
make by plotting it as we just did.

</details>